In [23]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori
from mlxtend.preprocessing import TransactionEncoder
from sklearn.preprocessing import OneHotEncoder
from mlxtend.frequent_patterns import association_rules
from ipywidgets import interact
import ipywidgets as widgets
from IPython.display import display

In [24]:
df = pd.read_csv("HR_comma_sep (1).csv")
df.head()

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.10,0.77,6,247,4,0,1,0,sales,low
3,0.92,0.85,5,259,5,0,1,0,sales,low
4,0.89,1.00,5,224,5,0,1,0,sales,low


In [25]:
df.describe()

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years
count,14995.000000,14995.000000,14995.000000,14995.000000,14995.000000,14995.000000,14995.000000,14995.000000
mean,0.612890,0.716108,3.803001,201.050150,3.498166,0.144648,0.237879,0.021274
std,0.248615,0.171166,1.232266,49.943356,1.460262,0.351758,0.425799,0.144300
min,0.090000,0.360000,2.000000,96.000000,2.000000,0.000000,0.000000,0.000000
25%,0.440000,0.560000,3.000000,156.000000,3.000000,0.000000,0.000000,0.000000
50%,0.640000,0.720000,4.000000,200.000000,3.000000,0.000000,0.000000,0.000000
75%,0.820000,0.870000,5.000000,245.000000,4.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,7.000000,310.000000,10.000000,1.000000,1.000000,1.000000


In [26]:
def bins(input,b):
    binned = pd.cut(input, bins = b)
    return binned

In [27]:
df['satisfaction'] = bins(df['satisfaction_level'],5)
df['evaluation'] = bins(df['last_evaluation'],5)
df['avg_hours'] = bins(df['average_montly_hours'],5)
df['project'] = bins(df['number_project'],3)
df['time_spend'] = bins(df['time_spend_company'],3)

In [28]:
df.drop(['satisfaction_level','last_evaluation','average_montly_hours','number_project','time_spend_company'],axis=1,inplace=True)

In [29]:
ohc = OneHotEncoder(sparse_output=False, dtype=np.bool_).set_output(transform='pandas')
df = ohc.fit_transform(df)

In [30]:
def gen_rules (min_support, min_confidence):
    itemsets = apriori(df, min_support=min_support, use_colnames=True)
    rules = association_rules(itemsets, metric='confidence', min_threshold=min_confidence)
    rules = rules[['antecedents','consequents','support','confidence','lift']]
    rel_rules = rules[rules["lift"]>1]
    return rel_rules.sort_values(by=['confidence','lift','support'],ascending=False)

In [31]:
gen_rules(0.3,0.4)

,antecedents,consequents,support,confidence,lift
182,"(Work_accident_0, time_spend_(1.992, 4.667], s...",(promotion_last_5years_0),0.341581,0.993213,1.014802
63,"(Work_accident_0, salary_low)",(promotion_last_5years_0),0.415138,0.992348,1.013918
135,"(time_spend_(1.992, 4.667], salary_low)",(promotion_last_5years_0),0.398866,0.991874,1.013434
27,(salary_low),(promotion_last_5years_0),0.483294,0.990975,1.012515
102,"(left_0, salary_low)",(promotion_last_5years_0),0.339580,0.989891,1.011408
...,...,...,...,...,...
195,"(promotion_last_5years_0, time_spend_(1.992, 4...","(Work_accident_0, project_(1.995, 3.667])",0.327776,0.408970,1.113179
136,(promotion_last_5years_0),"(time_spend_(1.992, 4.667], salary_low)",0.398866,0.407536,1.013434
183,"(promotion_last_5years_0, Work_accident_0)","(time_spend_(1.992, 4.667], salary_low)",0.341581,0.407057,1.012242
98,"(time_spend_(1.992, 4.667])","(Work_accident_0, project_(1.995, 3.667])",0.331911,0.406584,1.106686


In [ ]:
intervals = [(0,0.2),(0.2,0.4),(0.4,0.6),(0.6,0.8),(0.8,1.0)]
bins = pd.IntervalIndex.from_tuples(intervals)
df['satisfaction_level_slab'] = pd.cut(df['satisfaction_level'], bins)
df['last_evaluation_slab'] = pd.cut(df['last_evaluation'], bins)